In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [2]:
import sys, os

print(os.getpid())

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
add_path(os.environ['PROJECT'])

import project

import torch
assert torch.cuda.is_available()

1298332


# Gather examples

In [3]:
from pathlib import Path

#data_root = Path(os.environ['LP_ROOT']) / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'

subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = project.fileio.load_subject_list(subject_file, key='sid', sep='\t')

bad_subjects = ['23686J', '24634V']
subject_list = [sid for sid in subject_list if sid not in bad_subjects][:1]

len(subject_list)

Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.csv


1

In [4]:
config = project.fileio.load_config('2026-09-15_config.yaml')

data_config = config['dataset']
data_config['subjects'] = subject_list
data_config['pipeline_tags']['mesh_generation'] = 'rad'
data_config['pipeline_tags']['field_interpolation'] = 'int'
data_config['pipeline_tags']['forward_simulation'] = 'LE'

project.utils.pprint(data_config)

Loading 2026-09-15_config.yaml
dict(len=6)
├── 'name':          'COPDGene'
├── 'root':          '/restricted/projectnb/batmanlab/mragoza/data/COPDGene'
├── 'subjects':      list(len=1)
|   └── 0: '16514P'
├── 'state_pairs':   list(len=1)
|   └── 0: list(len=2)
├── 'variant':       '2026-09-15'
└── 'pipeline_tags': dict(len=8)
    ├── 'image_resampling':    'iso'
    ├── 'image_segmentation':  'v3d'
    ├── 'image_registration':  'gil'
    ├── 'anatomical_regions':  'lung'
    ├── 'material_properties': 'bela'
    ├── 'mesh_generation':     'rad'
    ├── 'field_interpolation': 'int'
    └── 'forward_simulation':  'LE'


In [5]:
examples = project.get_examples(data_config)

Gathering examples...
1 total


# Training run

In [6]:
train_config = config['training']
project.utils.pprint(train_config)

dict(len=9)
├── 'seed':       1234
├── 'task':       dict(len=3)
|   ├── 'inputs':  list(len=1)
|   ├── 'targets': list(len=1)
|   └── 'losses':  dict(len=1)
├── 'split':      dict(len=3)
|   ├── 'key':        'category'
|   ├── 'test_ratio': 0.2
|   └── 'val_ratio':  0.1
├── 'transform':  dict(len=8)
|   ├── 'normalize':      True
|   ├── 'image_mean':     0.0
|   ├── 'image_std':      1.0
|   ├── 'apply_mask':     True
|   ├── 'do_augment':     True
|   ├── 'rand_rotate':    True
|   ├── 'rand_reflect':   True
|   └── 'rand_translate': 4.0
├── 'loader':     dict(len=3)
|   ├── 'batch_size':  1
|   ├── 'shuffle':     True
|   └── 'num_workers': 0
├── 'model':      dict(len=2)
|   ├── 'backbone': dict(len=11)
|   └── 'heads':    dict(len=1)
├── 'pde_solver': dict(len=5)
|   ├── 'type':          'warp'
|   ├── 'material_type': 'linear'
|   ├── 'g_vector':      list(len=3)
|   ├── 'relative_loss': True
|   └── 'tv_reg_weight': 1e-05
├── 'optimizer':  dict(len=2)
|   ├── 'type': 'AdamW'
|

In [11]:
train_config = {
    'seed': 1234,
    'split': {'method': 'random', 'test_ratio': 0, 'val_ratio': 0},
    'transform': {
        'normalize': False,
        'apply_mask': True,
        'do_augment': False,
        'use_cache': True
    },
    'task': {
        'inputs': ['image'],
        'targets': ['E'],
        'losses': {'E': 'sim'}
    },
    'model': {
        'backbone': {
            'type': 'UNet3Dv2',
            'conv_channels': 8,
            'n_conv_units': 2,
            'n_sub_levels': 2
        },
        'heads': {
            'E': {
                'mode': 'log10', 'scale': 1.0, 'v_loc': 2e3, 'v_min': 0.0, 'v_max': 1e5
            }
        }
    },
    'optimizer': {'type': 'AdamW', 'lr': 1e-3},
    'boundary_condition': {'type': 'mesh_key', 'value': 'u_fwd'},
    'trainer': {'num_epochs': 100}
}

In [12]:
%ls checkpoints

checkpoint00000.pt


In [9]:
%rm -rf checkpoints

In [13]:
%autoreload
project.run_training(examples, train_config)

Splitting data examples
Train examples: 1
Test examples:  0
Val examples:   0


Validating training task
Inputs:  ['image']
Targets: ['E']
Losses:  {'E': 'sim'}
Building model architecture
MultiTaskModel(
  (backbone): UNet3Dv2(
    (enc): ConvBlock3D(
      (unit0): ConvUnit3D(
        (conv): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=same, bias=False, padding_mode=replicate)
        (norm): GroupNorm(4, 8, eps=1e-05, affine=True)
        (act): LeakyReLU(negative_slope=0.1, inplace=True)
      )
      (unit1): ConvUnit3D(
        (conv): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=same, bias=False, padding_mode=replicate)
        (norm): GroupNorm(4, 8, eps=1e-05, affine=True)
        (act): LeakyReLU(negative_slope=0.1, inplace=True)
      )
    )
    (net): UNet3Dv2(
      (enc): ConvBlock3D(
        (unit0): ConvUnit3D(
          (conv): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=same, bias=False, padding_mode=replicate)
          (norm): GroupNorm(4, 16, eps=1e-05, affine=True)
          (act): LeakyR